# signals

> Pure deterministic Tier-1 signal functions (no capability calls): empty-segment
> detection, bidirectional boundary punctuation/capitalization heuristics,
> forced-alignment coverage flags, positional cross-transcriber diff, and
> phonetic + edit-distance variant clustering. The worklist is recomputed from
> these each session; revolution-1 builds ZERO new capabilities.

In [ ]:
#| default_exp signals

In [ ]:
#| export
import re
from typing import Dict, List, Optional, Tuple

from cjm_transcript_correction_core.models import SpineSegment

In [ ]:
#| export
def detect_empty_segments(
    segments: List[SpineSegment],  # Ordered spine segments
) -> List[int]:  # Positions (in `segments`) of empty-text segments
    """Find empty-text segments (silence VAD chunks with no aligned words; decomp D14)."""
    return [i for i, s in enumerate(segments) if s.is_empty]

In [ ]:
#| export
_TERMINAL_PUNCT = (".", "!", "?", "。", "！", "？")  # incl. CJK full-stop / ! / ?


def _ends_terminal(text: str) -> bool:  # True if text ends with sentence-terminal punctuation
    """Whether a segment's text ends with terminal punctuation (trailing quotes/brackets ignored)."""
    t = (text or "").rstrip().rstrip("\"')”’")
    return t.endswith(_TERMINAL_PUNCT)


def _starts_upper(text: str) -> bool:  # True if the first alphabetic char is uppercase
    """Whether a segment's text starts with an uppercase letter (leading quotes/brackets ignored)."""
    for ch in (text or "").lstrip("\"'(“‘"):
        if ch.isalpha():
            return ch.isupper()
        if not ch.isspace():
            return False
    return False

In [ ]:
#| export
def boundary_punct_caps_flags(
    segments: List[SpineSegment],  # Ordered spine segments
) -> Dict[int, List[str]]:  # segment index -> boundary flags
    """Bidirectional boundary punctuation/capitalization heuristics (in-segment only).

    At each border (seg[i] -> seg[i+1]) flag the two error directions a downstream
    grouping workflow cares about, WITHOUT ever merging across audio segments:
      - "boundary-missing-terminal": seg[i] lacks terminal punctuation and seg[i+1]
        starts uppercase -> a sentence may end here but is missing a period.
      - "boundary-terminal-then-lowercase": seg[i] ends terminal but seg[i+1] starts
        lowercase -> one sentence may have been split across the border.
    Empty neighbours are skipped (handled by the prune).
    """
    flags: Dict[int, List[str]] = {}
    for i in range(len(segments) - 1):
        a, b = segments[i], segments[i + 1]
        if a.is_empty or b.is_empty:
            continue
        bt = b.text.strip()
        if not _ends_terminal(a.text) and _starts_upper(b.text):
            flags.setdefault(i, []).append("boundary-missing-terminal")
        if _ends_terminal(a.text) and bt[:1].isalpha() and not _starts_upper(b.text):
            flags.setdefault(i, []).append("boundary-terminal-then-lowercase")
    return flags

In [ ]:
#| export
def fa_coverage_flags(
    segments: List[SpineSegment],  # Ordered spine segments
) -> Dict[int, List[str]]:  # segment index -> coverage flags
    """Flag segments whose forced-alignment coverage looks suspect (Tier-1).

    Empty-text segments (no aligned words) and segments missing source-coordinate
    timing are flagged; both are alignment-failure signals shared by text and
    segmentation errors.
    """
    flags: Dict[int, List[str]] = {}
    for i, s in enumerate(segments):
        if s.is_empty:
            flags.setdefault(i, []).append("empty-text")
        if s.start_time is None or s.end_time is None:
            flags.setdefault(i, []).append("missing-timing")
    return flags

In [ ]:
#| export
def levenshtein(
    a: str,  # First string
    b: str,  # Second string
) -> int:  # Edit distance
    """Levenshtein edit distance (pure, in-core; variant-clustering primitive)."""
    a, b = a or "", b or ""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

In [ ]:
#| export
_WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)  # alphabetic word tokens (unicode-aware)


def phonetic_key(
    word: str,  # A single word token
) -> str:  # A coarse phonetic key (Soundex-like, in-core)
    """Compute a coarse phonetic key for a word (groups like-sounding variants).

    A lightweight Soundex-style reduction (first letter + consonant codes, vowels
    dropped): enough to bucket transcription variants of one entity for
    fix-one-fix-all, without a phonetics dependency.
    """
    w = "".join(ch for ch in (word or "").lower() if ch.isalpha())
    if not w:
        return ""
    codes = {**dict.fromkeys("bfpv", "1"), **dict.fromkeys("cgjkqsxz", "2"),
             **dict.fromkeys("dt", "3"), **dict.fromkeys("l", "4"),
             **dict.fromkeys("mn", "5"), **dict.fromkeys("r", "6")}
    first = w[0]
    tail: List[str] = []
    prev = codes.get(first, "")
    for ch in w[1:]:
        c = codes.get(ch, "")
        if c and c != prev:
            tail.append(c)
        prev = c
    return (first + "".join(tail) + "000")[:4]

In [ ]:
#| export
def _normalize_text(text: str) -> str:  # Lowercased alphabetic word tokens, space-joined
    """Normalize segment text for cross-transcriber comparison."""
    return " ".join(_WORD_RE.findall((text or "").lower()))


def cross_transcriber_diff(
    primary: List[SpineSegment],    # Primary (e.g. accuracy-model) spine, ordered
    secondary: List[SpineSegment],  # Secondary (e.g. lightweight-model) spine, ordered
) -> Dict[int, Tuple[str, str]]:  # primary index -> (primary_text, secondary_text) where they diverge
    """Positionally diff two transcriber spines of the SAME source.

    Both decomp spines share an identical VAD-chunk skeleton (same audio -> same
    VAD timing), so they align 1:1 by index; proper-noun / error sites concentrate
    where the normalized texts diverge (the force-multiplier signal). A length
    mismatch is recorded at the sentinel key -1.
    """
    diffs: Dict[int, Tuple[str, str]] = {}
    n = min(len(primary), len(secondary))
    if len(primary) != len(secondary):
        diffs[-1] = (f"len={len(primary)}", f"len={len(secondary)}")
    for i in range(n):
        pt, st = primary[i].text or "", secondary[i].text or ""
        if _normalize_text(pt) != _normalize_text(st):
            diffs[i] = (pt, st)
    return diffs

In [ ]:
#| export
def cluster_variants(
    words: List[str],    # Candidate word tokens (e.g. divergent proper nouns)
    max_edits: int = 2,  # Max edit distance to join two words into one cluster
) -> List[List[str]]:  # Clusters (size > 1) of like-sounding / near-spelled variants
    """Cluster word variants by phonetic key + edit distance (fix-one-fix-all).

    Buckets transcription variants of one entity so a single decision can map them
    all to a canonical form. Pure, in-core (no phonetics dependency).
    """
    uniq = list(dict.fromkeys(w.strip() for w in words if w and w.strip()))
    clusters: List[List[str]] = []
    keys: List[str] = []
    for w in uniq:
        k = phonetic_key(w)
        placed = False
        for ci, ck in enumerate(keys):
            if k and k == ck and levenshtein(w.lower(), clusters[ci][0].lower()) <= max_edits:
                clusters[ci].append(w)
                placed = True
                break
        if not placed:
            clusters.append([w])
            keys.append(k)
    return [c for c in clusters if len(c) > 1]

In [ ]:
#| export
def compute_signal_flags(
    segments: List[SpineSegment],                    # Ordered primary spine
    secondary: Optional[List[SpineSegment]] = None,  # Optional second-transcriber spine
) -> Dict[int, List[str]]:  # segment index -> combined Tier-1 flags
    """Combine all deterministic Tier-1 signals into per-segment flags.

    The worklist is RECOMPUTED from this each session (only decisions persist);
    new signals join here and are picked up automatically.
    """
    flags: Dict[int, List[str]] = {}

    def add(idx: int, fl: List[str]) -> None:
        bucket = flags.setdefault(idx, [])
        for f in fl:
            if f not in bucket:
                bucket.append(f)

    for idx, fl in fa_coverage_flags(segments).items():
        add(idx, fl)
    for idx, fl in boundary_punct_caps_flags(segments).items():
        add(idx, fl)
    if secondary is not None:
        for idx in cross_transcriber_diff(segments, secondary):
            if idx >= 0:
                add(idx, ["transcriber-divergence"])
    return flags

In [ ]:
# signals smoke checks
segs = [
    SpineSegment(id="0", index=0, text="The art of war", start_time=0.0, end_time=1.0),
    SpineSegment(id="1", index=1, text="", start_time=1.0, end_time=1.2),
    SpineSegment(id="2", index=2, text="is of vital importance.", start_time=1.2, end_time=2.0),
    SpineSegment(id="3", index=3, text="the general who wins", start_time=2.0, end_time=3.0),
]
assert detect_empty_segments(segs) == [1]
assert "empty-text" in fa_coverage_flags(segs)[1]
# 2->3: "...importance." terminal, "the general..." lowercase -> terminal-then-lowercase
b = boundary_punct_caps_flags(segs)
assert "boundary-terminal-then-lowercase" in b.get(2, [])
assert levenshtein("nickel", "nccl") >= 1
assert phonetic_key("nickel") == phonetic_key("nichol")  # like-sounding bucket
prim = [SpineSegment(id="a", index=0, text="ChatGPT is great")]
sec = [SpineSegment(id="b", index=0, text="Chachi PT is great")]
assert 0 in cross_transcriber_diff(prim, sec)
assert isinstance(cluster_variants(["ChatGPT", "Chachi", "unrelated"]), list)
flags = compute_signal_flags(segs, secondary=[
    SpineSegment(id="x0", index=0, text="The art of war"),
    SpineSegment(id="x1", index=1, text=""),
    SpineSegment(id="x2", index=2, text="is of VITAL stuff."),
    SpineSegment(id="x3", index=3, text="the general who wins"),
])
assert 1 in flags and "transcriber-divergence" in flags.get(2, [])
print("signals checks OK")

signals checks OK
